In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score
import pandas as pd

from Dataloader_Excel import read_excel_eeg
from EEGNet_EF import EEGNet as net

# Hyperparameters
depth = 4
depth_series = [4, 4]
query_size = 64
key_size = 64
value_size = 64
emb_size = 64
num_heads = 4
expansion = 2
conv_dropout = 0.3
self_dropout = 0.3
cross_dropout = 0.3
cls_dropout = 0.5
num_classes = 2 # number of paradigm
batch_size = 16
num_epochs = 200
learning_rate = 0.001

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Create directories for results
os.makedirs("Results", exist_ok=True)

# Initialize model inside the fold loop to prevent data leak
model = net().to(device, dtype=torch.float64)

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

def evaluate_model(eval_loader, model):
    model.eval()
    total_loss, total_correct = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for eval_eeg, eval_labels in eval_loader:
            eval_eeg = eval_eeg.to(device, dtype=torch.float64)
            eval_labels = eval_labels.to(device, dtype=torch.long)

            outputs = model(eval_eeg)
            loss = criterion(outputs, eval_labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            total_correct += (preds == eval_labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(eval_labels.cpu().numpy())

    loss = total_loss / len(eval_loader)
    acc = total_correct / len(eval_loader.dataset)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    kappa = cohen_kappa_score(all_labels, all_preds)

    print(f"Evaluation Loss: {loss:.2f} | Evaluation Acc: {acc:.2f}")
    return acc, precision, recall, f1, kappa

for subject_id in range(29):
    subject_id += 1
    eeg, labels = read_excel_eeg(subject_id, mode=0) # (60, 30, 7000), (60,)
    eeg = eeg.permute(0, 2, 1)
    eeg = eeg.unsqueeze(1)
    eeg, labels = eeg.to(device), labels.to(device)

    train_size = int(0.6*len(eeg))
    eval_size = len(eeg) - train_size

    # create a tensor dataset
    dataset = torch.utils.data.TensorDataset(eeg, labels)

    # randomly split the data into training and evaluation sets
    train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        total_correct, total_loss = 0, 0

        for batch_eeg, batch_labels in train_loader:
            batch_eeg = batch_eeg.to(device, dtype=torch.float64)
            batch_labels = batch_labels.to(device, dtype=torch.long)

            optimizer.zero_grad()
            outputs = model(batch_eeg)
            preds = torch.argmax(outputs, dim=1)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_correct += (preds == batch_labels).sum().item()

        # len(train_loader) is the number of batches
        # len(train_dataset) is the number of trials
        loss = total_loss / len(train_loader)
        acc = total_correct / len(train_loader.dataset)
        print(f"Epoch {epoch} Training Loss: {loss:.2f} | Training Acc: {acc:.2f}")

        acc, precision, recall, f1, kappa = evaluate_model(eval_loader, model)
        acc_list.append(acc)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        kappa_list.append(kappa)

    mean_acc, std_acc = np.mean(acc_list[-50:]), np.std(acc_list[-50:])
    mean_precision, std_precision = np.mean(precision_list[-50:]), np.std(precision_list[-50:])
    mean_recall, std_recall = np.mean(recall_list[-50:]), np.std(recall_list[-50:])
    mean_f1, std_f1 = np.mean(f1_list[-50:]), np.std(f1_list[-50:])
    mean_kappa = np.mean(kappa_list[-50:])

    mean_acc, std_acc = mean_acc*100, std_acc*100
    mean_precision, std_precision = mean_precision*100, std_precision*100
    mean_recall, std_recall = mean_recall*100, std_recall*100
    mean_f1, std_f1 = mean_f1*100, std_f1*100

    log_path = f'Results/log_s{subject_id:2d}.txt'
    log_excel = f'Results/Log.xlsx'
    with open(log_path, 'a') as log_file:
        log_file.write(f'Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')
        log_file.write(f'Precision: {mean_precision:.2f} ± {std_precision:.2f}\n')
        log_file.write(f'Recall: {mean_recall:.2f} ± {std_recall:.2f}\n')
        log_file.write(f'F1: {mean_f1:.2f} ± {std_f1:2f}\n')
        log_file.write(f'Kappa: {mean_kappa:.2f}\n')
        log_file.write('\n')

    new_row = pd.DataFrame([[mean_acc, std_acc, mean_precision, std_precision, mean_recall, std_recall, mean_f1, std_f1, mean_kappa]], 
                           columns=['Accuracy', 'Std_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
    new_row = new_row.round(2)
    if not os.path.exists(log_excel):
        new_row.to_excel(log_excel, index=False, engine='openpyxl')
    else:
        with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
            if 'Sheet1' in writer.sheets:
                startrow = writer.sheets['Sheet1'].max_row
            else:
                startrow = 0
            new_row.to_excel(writer, index=False, header=False, startrow=startrow)

    print(f'Subject:{subject_id} Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')

print('Training and evaluation completed\n')

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


1 EEG torch.Size([60, 30, 4000]) Labels torch.Size([60])
Epoch 0 Training Loss: 0.73 | Training Acc: 0.53
Evaluation Loss: 0.80 | Evaluation Acc: 0.46
Epoch 1 Training Loss: 0.69 | Training Acc: 0.53
Evaluation Loss: 0.80 | Evaluation Acc: 0.46


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2 Training Loss: 0.66 | Training Acc: 0.53
Evaluation Loss: 0.73 | Evaluation Acc: 0.50
Epoch 3 Training Loss: 0.61 | Training Acc: 0.72
Evaluation Loss: 0.66 | Evaluation Acc: 0.50
Epoch 4 Training Loss: 0.57 | Training Acc: 0.86
Evaluation Loss: 0.62 | Evaluation Acc: 0.67
Epoch 5 Training Loss: 0.56 | Training Acc: 0.94
Evaluation Loss: 0.61 | Evaluation Acc: 0.71
Epoch 6 Training Loss: 0.49 | Training Acc: 0.94
Evaluation Loss: 0.61 | Evaluation Acc: 0.75
Epoch 7 Training Loss: 0.46 | Training Acc: 0.86
Evaluation Loss: 0.60 | Evaluation Acc: 0.71
Epoch 8 Training Loss: 0.41 | Training Acc: 0.92
Evaluation Loss: 0.58 | Evaluation Acc: 0.75
Epoch 9 Training Loss: 0.41 | Training Acc: 0.94
Evaluation Loss: 0.57 | Evaluation Acc: 0.75
Epoch 10 Training Loss: 0.41 | Training Acc: 0.94
Evaluation Loss: 0.58 | Evaluation Acc: 0.79
Epoch 11 Training Loss: 0.38 | Training Acc: 0.92
Evaluation Loss: 0.58 | Evaluation Acc: 0.71
Epoch 12 Training Loss: 0.44 | Training Acc: 0.92
Evaluati

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 86 Training Loss: 0.59 | Training Acc: 0.69
Evaluation Loss: 0.79 | Evaluation Acc: 0.54
Epoch 87 Training Loss: 0.66 | Training Acc: 0.72
Evaluation Loss: 0.78 | Evaluation Acc: 0.50


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 88 Training Loss: 0.60 | Training Acc: 0.72
Evaluation Loss: 0.78 | Evaluation Acc: 0.50
Epoch 89 Training Loss: 0.50 | Training Acc: 0.78
Evaluation Loss: 0.77 | Evaluation Acc: 0.50


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 90 Training Loss: 0.58 | Training Acc: 0.72
Evaluation Loss: 0.78 | Evaluation Acc: 0.50
Epoch 91 Training Loss: 0.57 | Training Acc: 0.72
Evaluation Loss: 0.78 | Evaluation Acc: 0.54
Epoch 92 Training Loss: 0.51 | Training Acc: 0.72
Evaluation Loss: 0.77 | Evaluation Acc: 0.54
Epoch 93 Training Loss: 0.63 | Training Acc: 0.72
Evaluation Loss: 0.78 | Evaluation Acc: 0.50
Epoch 94 Training Loss: 0.51 | Training Acc: 0.72
Evaluation Loss: 0.77 | Evaluation Acc: 0.54
Epoch 95 Training Loss: 0.57 | Training Acc: 0.72
Evaluation Loss: 0.79 | Evaluation Acc: 0.50
Epoch 96 Training Loss: 0.62 | Training Acc: 0.72
Evaluation Loss: 0.82 | Evaluation Acc: 0.46
Epoch 97 Training Loss: 0.70 | Training Acc: 0.72
Evaluation Loss: 0.79 | Evaluation Acc: 0.50
Epoch 98 Training Loss: 0.57 | Training Acc: 0.72
Evaluation Loss: 0.77 | Evaluation Acc: 0.46
Epoch 99 Training Loss: 0.57 | Training Acc: 0.72
Evaluation Loss: 0.80 | Evaluation Acc: 0.50
Epoch 100 Training Loss: 0.59 | Training Acc: 0.72